# RunResult, límites y preparación

Resultado serializable y linaje en todos los límites de ejecución.

API **2.1.3**, kernel dedicado y ejecución offline. Los actores Python son operadores deterministas.

In [1]:
from pathlib import Path
import sys
tutorial_root = next(p / "tutorials" for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "tutorials/_offline.py").is_file())
sys.path.insert(0, str(tutorial_root))
from _offline import enforce_offline, blocked_attempts
enforce_offline()
import agentic_systems as toolkit
assert toolkit.__version__ == "2.1.3"
assert "site-packages" in Path(toolkit.__file__).parts
sys.path.insert(0, str(tutorial_root.parent / "examples"))


In [2]:
@toolkit.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@toolkit.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = toolkit.skill(name="arithmetic", tools=[increment, double])
increase = toolkit.Action("increase", "increment")
multiply = toolkit.Action("multiply", "double")

In [3]:
import json
agent = toolkit.Agent("calculator", toolkit.Sequence((increase, multiply)), skills=(arithmetic,))
compiled = agent.compile()
result = compiled.run({"value": 3}, limits=toolkit.Limits(max_actions=2, timeout_s=10))
assert result.ok
encoded = json.loads(json.dumps(result.to_dict(), allow_nan=False))
assert toolkit.RunResult(**encoded).data == {"value": 8}
for node in result.walk():
    for child in node.children:
        assert child.parent_execution_id == node.execution_id
limited = agent.run({"value": 3}, limits=toolkit.Limits(max_actions=1))
assert not limited.ok
print({"output": result.data, "lineage_nodes": len(list(result.walk())), "limit_detected": not limited.ok})

{'output': {'value': 8}, 'lineage_nodes': 6, 'limit_detected': True}


`compile()` no ejecuta Tools ni condiciones. `run()` lo llama internamente. Ambos caminos devuelven el mismo contrato; compile es preparación opcional, no otro constructor. Los límites son cooperativos y comparten reservas entre componentes; no matan un callback Python bloqueado. Agent/System también ofrecen `arun` con el mismo resultado.

In [4]:
assert (await agent.arun({'value':3})).data == result.data
assert (await compiled.arun({'value':3})).data == result.data
assert result.check_invariants().ok
assert result.trace('full') == result.to_dict()
memory = result.lineage(name='calculator',question='What happened?',goal='Explain',tags=['offline'],metadata={'tutorial':7},max_tool_rows=2)
print(memory.explain())
print(memory.to_prompt_context(max_chars=500))
print(result.trace())

{'answer': '{"value": 8}', 'ok': True, 'steps': [{'step_id': 'run-ed36b6c2808c4be8a14b9c9fb031dce8', 'kind': 'execution', 'title': 'calculator', 'summary': '', 'source': 'python-runtime', 'why': 'Recorded execution evidence', 'evidence': {'ok': True, 'data': {'value': 8}, 'errors': []}}, {'step_id': 'run-bdb47af4b36e491b8eea9ec411bf1134', 'kind': 'execution', 'title': 'sequence', 'summary': '', 'source': 'python-runtime', 'why': 'Recorded execution evidence', 'evidence': {'ok': True, 'data': {'value': 8}, 'errors': []}}, {'step_id': 'run-2c700d0b65174cc89f853b19ea2dd20c', 'kind': 'execution', 'title': 'increase', 'summary': '', 'source': 'python-runtime', 'why': 'Recorded execution evidence', 'evidence': {'ok': True, 'data': {'value': 4}, 'errors': []}}, {'step_id': 'run-7fb8dfb89a354c58bd46033227bd7607', 'kind': 'execution', 'title': 'tool', 'summary': '', 'source': 'python-runtime', 'why': 'Recorded execution evidence', 'evidence': {'ok': True, 'data': {'value': 4}, 'errors': []}}, {

to_dict es el esquema de intercambio; RunResult(**payload) restaura ese contrato. walk recorre la jerarquía, trace resume ejecución y lineage proyecta evidencia para explicación. Los contadores de Tools e inferencia son distintos. raw_responses no contiene objetos SDK ni credenciales. Los timeouts son cooperativos, no cancelación fuerte.

In [5]:
assert blocked_attempts() == 0
print("PASS · inferencia: 0")

PASS · inferencia: 0
